# **Notebook 4: RAG Implementation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] `corporate_policies/` folder with `.md` SOP files
- [ ] `outputs.json` — Created by Notebook 3
- [ ] GPU runtime enabled

**Files this notebook will CREATE:**
- [ ] `./chroma_db/` — Persisted ChromaDB vector index _(Required by NB5 and NB7)_
- [ ] `outputs.json` (updated) — adds `naive_rag_output` _(Required by NB5 and NB7)_

---

### **Task 3.2: Implement Retrieval-Assisted Generation**

#### **3.2.1 Generate Embeddings [4 marks]**
**The Task:** Initialise the `all-MiniLM-L6-v2` embedding model, embed the SOP documents, and validate the embeddings were produced.

**Hints & Tips:**
* Load the SOP documents with `TextLoader` first (or reuse the corpus from NB2).
* `HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")` runs on CPU — no GPU needed.
* Validate by embedding one sample string and checking the vector length (384 dims for MiniLM).
* You MUST use the same embedding model when reloading in Notebooks 5 and 7.

**Embedding Model Options:**
* **`all-MiniLM-L6-v2`** (recommended): 384-dim, fast, ~80MB.
* **`all-mpnet-base-v2`**: 768-dim, higher quality, slower.
* **`bge-small-en-v1.5`**: 384-dim, newer architecture.

**Learner Inference:** Your text is now coordinates in semantic space — similar meanings sit close together.

In [1]:
%pip install -q langchain-community langchain-huggingface sentence-transformers chromadb

Note: you may need to restart the kernel to use updated packages.


In [2]:
# YOUR CODE HERE
# --- 1. Load the SOP documents ---
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_huggingface import HuggingFaceEmbeddings

loader = DirectoryLoader(
    "Dataset/sop_documents/",          
    glob="**/*.md",                 
    loader_cls=TextLoader,          
)
sop_docs = loader.load()
print(f"Loaded {len(sop_docs)} SOP documents")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

sample_vector = embeddings.embed_query("I want a refund")
print(f"Vector length: {len(sample_vector)}")
print(f"First 5 numbers: {sample_vector[:5]}")

/var/folders/96/jylj5rkx5n13cx5tm6mgdjc80000gn/T/ipykernel_2394/2836465858.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader
/Users/abinas/Documents/projects/machine-learning/capstone-project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 13 SOP documents


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 33814.44it/s]


Vector length: 384
First 5 numbers: [-0.07807613909244537, 0.01004492025822401, 0.030082592740654945, 0.025594554841518402, 0.017350943759083748]


#### **3.2.2 Build Vector Index [4 marks]**
**The Task:** Create a persistent Chroma vector index from the embedded documents, configure similarity search, and validate the index.

**Hints & Tips:**
* `Chroma.from_documents(docs, embeddings, persist_directory="./chroma_db")` auto-saves — no manual `.persist()` needed.
* Validate with `vector_db._collection.count()` — should equal the number of SOP documents.
* Run one test `.similarity_search("refund", k=1)` to confirm retrieval works.

**Vector DB Options:**
* **ChromaDB** (recommended): simple API, auto-persistence, LangChain integration.
* **FAISS**: faster for >100K docs, but no built-in persistence (manual serialization).

**Learner Inference:** The index lets you search by meaning — it returns the document mathematically closest to your query's coordinates.

In [3]:
# YOUR CODE HERE

from langchain_community.vectorstores import Chroma

vector_db = Chroma.from_documents(
    documents=sop_docs,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

count = vector_db._collection.count()
print(f"Documents in index: {count}")

result = vector_db.similarity_search("refund", k=1)
print(f"Top match source: {result[0].metadata['source']}")
print(f"Preview: {result[0].page_content[:500]}")


Documents in index: 13
Top match source: Dataset/sop_documents/refund_policy.md
Preview: # Refund Policy

## Eligibility
Customers may request a refund within 30 days of the original purchase or
delivery date, whichever is later. To be eligible, the item must be unused or
defective, and the request must reference a valid order identifier. Digital
goods and subscription fees already consumed are non-refundable except where
required by local consumer law.

## Refund Methods
Approved refunds are issued to the original payment method. If the original
card or wallet is no longer active, 


#### **3.2.3 Implement Retrieval Workflow [4 marks]**
**The Task:** Execute a "Naive RAG" workflow — pass the raw customer query into the vector DB, fetch the top result, and augment the LLM prompt.

**Hints & Tips:**
* Use `.similarity_search(query, k=1)` for the top-1 document.
* Check whether the raw query retrieved the WRONG policy — common with ambiguous queries.
* Inject context via the system prompt: `"Answer strictly using this SOP: {context}"`.

**Parameter Tuning:**
* `k=1`: one document (focused). `k=3`: more context if SOPs overlap. `k=5`: max, risks long prompts.

**Learner Inference:** Noisy queries often retrieve the wrong document — proving Naive RAG is flawed and motivating the fine-tuned router.

In [6]:

import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

with open("outputs.json", "r") as f:
    outputs = json.load(f)

test_query = outputs["test_query"]
print(f"Test query: {test_query}")


MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
device = "mps" if torch.backends.mps.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
).to(device)
print(f"Model loaded on: {device}")

Test query: My order is late, where is it??


Loading weights: 100%|██████████| 338/338 [00:01<00:00, 182.51it/s]


Model loaded on: mps


In [8]:
# YOUR CODE HERE
retrieved = vector_db.similarity_search(test_query, k=1)
context = retrieved[0].page_content
source = retrieved[0].metadata["source"]

print(f"Retrieved SOP: {source}\n")

messages = [
    {"role": "system", "content": f"You are a customer support agent. Answer strictly using this SOP:\n\n{context}"},
    {"role": "user", "content": test_query},
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(prompt, return_tensors="pt").to(device)

output_ids = model.generate(
    **inputs,
    max_new_tokens=200,
    do_sample=False,
    temperature=None,
    top_p=None,
)

new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
naive_rag_output = tokenizer.decode(new_tokens, skip_special_tokens=True)

print("--- NAIVE RAG OUTPUT ---")
print(naive_rag_output)

Retrieved SOP: Dataset/sop_documents/shipping_delays.md

--- NAIVE RAG OUTPUT ---
I'm sorry to hear that your order is delayed. To help you better, could you please provide me with the order number? Additionally, I would like to know if there were any specific reasons mentioned when the order was placed regarding the expected delivery time. This information will assist me in providing you with the most accurate response possible.


---
## Save Artifacts for Downstream Notebooks

In [9]:
# YOUR CODE HERE
# --- 1. Add the new result to the existing outputs dict ---
import json

outputs["naive_rag_output"] = naive_rag_output

with open("outputs.json", "w") as f:
    json.dump(outputs, f, indent=2)

print("Saved naive_rag_output to outputs.json")

with open("outputs.json", "r") as f:
    check = json.load(f)

print("\nKeys in outputs.json:", list(check.keys()))
print("\nnaive_rag_output present:", "naive_rag_output" in check)

Saved naive_rag_output to outputs.json

Keys in outputs.json: ['test_query', 'ground_truth', 'baseline_output', 'naive_rag_output']

naive_rag_output present: True


---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 5.**

- [ ] SOP documents loaded via `TextLoader`
- [ ] **Embeddings generated and validated** ← _Task 3.2.1_
- [ ] **ChromaDB index built, validated, and persisted** ← _Task 3.2.2_
- [ ] Naive similarity search executed on `test_query`
- [ ] Naive RAG output generated with SOP context injected
- [ ] **`./chroma_db/` exists on disk** ← _CRITICAL for NB5 and NB7_
- [ ] **`outputs.json` updated** with `naive_rag_output` ← _CRITICAL for NB5 and NB7_

**If any item is unchecked, fix it before moving on.**